<a href="https://colab.research.google.com/github/lakshiyaaa2025-maker/lakshiya/blob/main/Probabilistic_SMS_Spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Probabilistic SMS Spam Detection
**Course:** BAMAT207 – Probability and Statistics  
**Project Activity:** Probability and Random Variables

## Aim
To develop a Python-based system that classifies SMS messages as **Spam** or **Genuine (Ham)** using probability concepts and a Multinomial Naive Bayes classifier.

## Objectives
- Load, clean, and analyse the SMS Spam Collection dataset.
- Calculate prior and word-based conditional probabilities.
- Verify a conditional probability using Bayes' theorem.
- Calculate the expectation and variance of a binary random variable.
- Train and evaluate a Multinomial Naive Bayes model.
- Predict whether a newly entered SMS is spam and display prediction probabilities.

> Run the notebook from top to bottom. The dataset counts, probabilities, charts, and evaluation scores will be calculated from the dataset when you run it; they are not hard-coded.

## 1. Install and import the required libraries
Run the installation cell once in Google Colab. If a package is already installed, pip will simply confirm or update it.

In [ ]:
%pip -q install ucimlrepo

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

## 2. Load the SMS Spam Collection dataset
The assignment specifies the UCI SMS Spam Collection dataset (5,574 labelled messages). This cell downloads it through `ucimlrepo`.

In [ ]:
import pandas as pd
import zipfile
import io
import urllib.request

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

# Download the dataset
response = urllib.request.urlopen(url)
zip_data = zipfile.ZipFile(io.BytesIO(response.read()))

# Read only the SMS messages file
with zip_data.open("SMSSpamCollection") as file:
    df = pd.read_csv(
        file,
        sep="\t",
        header=None,
        names=["label", "message"],
        encoding="utf-8"
    )

print("Dataset loaded successfully!")
print("Number of messages:", len(df))
display(df.head())
print(df["label"].value_counts())

### Prepare the message and label columns
This step handles the dataset's column names and converts labels to the required numeric format: **Spam = 1, Ham = 0**.

In [ ]:
# Clean and prepare the dataset
df = df.dropna(subset=["label", "message"]).copy()

df["label"] = df["label"].astype(str).str.strip().str.lower()
df["message"] = df["message"].astype(str).str.strip()

# Keep only valid ham and spam labels
df = df[df["label"].isin(["ham", "spam"])].copy()

# Separate messages (X) and labels (y)
X_raw = df["message"]
y_raw = df["label"]

print("Dataset ready!")
print("Total messages:", len(df))
print("\nLabel counts:")
print(y_raw.value_counts())

Calculate the probability of spam

In [ ]:
# Calculate prior probabilities
total_messages = len(y_raw)

spam_count = (y_raw == "spam").sum()
ham_count = (y_raw == "ham").sum()

P_spam = spam_count / total_messages
P_ham = ham_count / total_messages

print(f"Total messages: {total_messages}")
print(f"Spam messages: {spam_count}")
print(f"Ham messages: {ham_count}")
print(f"P(Spam) = {P_spam:.4f} ({P_spam * 100:.2f}%)")
print(f"P(Ham)  = {P_ham:.4f} ({P_ham * 100:.2f}%)")

## 3. Dataset analysis and class distribution
Calculate the number of spam and genuine messages, their probabilities, and display a bar chart.

In [ ]:
class_counts = df["label"].value_counts().reindex(["spam", "ham"], fill_value=0)
total_messages = int(class_counts.sum())

p_spam = class_counts["spam"] / total_messages
p_ham = class_counts["ham"] / total_messages

print(f"Total messages: {total_messages}")
print(f"Spam messages: {int(class_counts['spam'])}")
print(f"Ham messages: {int(class_counts['ham'])}")
print(f"P(Spam) = {int(class_counts['spam'])} / {total_messages} = {p_spam:.4f}")
print(f"P(Ham)  = {int(class_counts['ham'])} / {total_messages} = {p_ham:.4f}")
print(f"Check: P(Spam) + P(Ham) = {p_spam + p_ham:.4f}")

plt.figure(figsize=(6, 4))
plt.bar(["Spam", "Ham"], [class_counts["spam"], class_counts["ham"]])
plt.title("Class Distribution of SMS Messages")
plt.xlabel("Message class")
plt.ylabel("Number of messages")
plt.tight_layout()
plt.show()

## 4. Word-based probability analysis
For each suspicious word (`free`, `prize`, `win`, `urgent`, `offer`), calculate:

\[
P(Spam\mid Word)=\frac{\text{Spam messages containing the word}}{\text{All messages containing the word}}
\]

The matching is case-insensitive and checks whether the word occurs as a whole word, so `win` does not count as part of `window`.

In [ ]:
suspicious_words = ["free", "prize", "win", "urgent", "offer"]

for word in suspicious_words:
    pattern = rf"\b{word}\b"

    contains_word = df["message"].str.contains(
        pattern, case=False, regex=True, na=False
    )

    total_with_word = int(contains_word.sum())

    spam_with_word = int(
        ((df["label"] == "spam") & contains_word).sum()
    )

    ham_with_word = int(
        ((df["label"] == "ham") & contains_word).sum()
    )

    p_spam_given_word = (
        spam_with_word / total_with_word
        if total_with_word > 0 else 0
    )

    print(f"Word: {word}")
    print(f"Messages containing word: {total_with_word}")
    print(f"Spam messages containing word: {spam_with_word}")
    print(f"Ham messages containing word: {ham_with_word}")
    print(f"P(Spam | {word}) = {p_spam_given_word:.4f}")
    print("-" * 35)

### Verify Bayes' theorem for the word “free”

In [ ]:
# Bayes' theorem for the word "free"

word = "free"

contains_free = df["message"].str.contains(
    rf"\b{word}\b",
    case=False,
    regex=True,
    na=False
)

# Prior probabilities
P_spam = (df["label"] == "spam").mean()
P_ham = (df["label"] == "ham").mean()

# Conditional probabilities
P_free_given_spam = contains_free[df["label"] == "spam"].mean()
P_free_given_ham = contains_free[df["label"] == "ham"].mean()

# Total probability of the word "free"
P_free = (
    P_free_given_spam * P_spam
    + P_free_given_ham * P_ham
)

# Bayes' theorem
P_spam_given_free = (
    P_free_given_spam * P_spam / P_free
    if P_free > 0 else 0
)

print(f"P(Spam) = {P_spam:.4f}")
print(f"P(Free | Spam) = {P_free_given_spam:.4f}")
print(f"P(Free | Ham) = {P_free_given_ham:.4f}")
print(f"P(Free) = {P_free:.4f}")
print(f"P(Spam | Free) = {P_spam_given_free:.4f}")
print(f"P(Spam | Free) = {P_spam_given_free * 100:.2f}%")

## 5. Random variable analysis
Define \(Y=1\) for spam and \(Y=0\) for ham. This is a Bernoulli random variable.

\[
E(Y)=P(Spam)=p
\]

\[
Var(Y)=p(1-p)
\]

In [ ]:
p = p_spam
expected_value = p
variance = p * (1 - p)

print(f"p = P(Spam) = {p:.4f}")
print(f"E(Y) = p = {expected_value:.4f}")
print(f"Var(Y) = p(1-p) = {variance:.4f}")

## 6. Data preparation
Split the data into 80% training and 20% testing, using stratified sampling to preserve the spam/ham proportion. Fit the text vectorizer using training messages only to avoid data leakage.

In [ ]:
# Data Preparation: split data and convert SMS text into features

# Input messages
X = df["message"].astype(str)

# Convert labels: spam = 1, ham = 0
y = df["label"].map({"spam": 1, "ham": 0})

# Remove any rows with labels that could not be mapped
valid_rows = y.notna()
X = X[valid_rows]
y = y[valid_rows].astype(int)

# Split into training and testing sets (80% / 20%)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Convert text messages into word-count features
vectorizer = CountVectorizer(lowercase=True, stop_words=None)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# Display results
print("Training messages:", len(X_train_text))
print("Testing messages:", len(X_test_text))
print("Training feature matrix shape:", X_train.shape)
print("Testing feature matrix shape:", X_test.shape)

print("\nLabel mapping: spam = 1, ham = 0")
print("\nTraining label distribution:")
print(y_train.value_counts().rename(index={1: "spam", 0: "ham"}))

## 7. Build the Multinomial Naive Bayes model
`CountVectorizer` converts text into word-count features. `MultinomialNB` learns patterns in these features to classify messages.

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

print("Model training completed.")

## 8. Model evaluation
Evaluate the test data using accuracy, precision, recall, F1-score, a classification report, and a confusion matrix.

In the confusion matrix:
- **Ham predicted as Spam** means a genuine message was incorrectly flagged as spam.
- **Spam predicted as Ham** means a spam message was missed.

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Ham", "Spam"],
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Ham", "Spam"])
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Correctly classified Ham (TN): {tn}")
print(f"Genuine messages incorrectly classified as Spam (FP): {fp}")
print(f"Spam messages incorrectly classified as Ham (FN): {fn}")
print(f"Correctly classified Spam (TP): {tp}")

## 9. Real-time testing
Enter a new SMS message in the input cell. The model will display the predicted class and the estimated probability of Spam and Ham.

In [ ]:
new_sms = input("Enter an SMS message to classify: ")

new_sms_features = vectorizer.transform([new_sms])
prediction = int(model.predict(new_sms_features)[0])
class_probabilities = model.predict_proba(new_sms_features)[0]

# Find the probability positions using the trained model's class labels.
class_to_index = {int(label): i for i, label in enumerate(model.classes_)}
p_ham_message = float(class_probabilities[class_to_index[0]])
p_spam_message = float(class_probabilities[class_to_index[1]])

print("\nPrediction:", "Spam" if prediction == 1 else "Genuine message (Ham)")
print(f"P(Spam | Message) = {p_spam_message:.4f}")
print(f"P(Ham | Message) = {p_ham_message:.4f}")

### Example test message
You can enter:

`Congratulations! You have won a free prize. Claim now.`

The predicted class and probability values depend on the trained model and dataset. Do not write the sample probabilities from the assignment as actual results unless your notebook produces those values.

## 10. Results and interpretation
After running all cells, use the actual values printed by your notebook to complete this summary.

- **Dataset summary:** Record the cleaned number of messages and the number of duplicates removed.
- **Class probabilities:** Record `P(Spam)` and `P(Ham)` from Activity 3.
- **Word probabilities:** Use the table from Activity 4 for all five suspicious words.
- **Random variable:** Record `E(Y)` and `Var(Y)` from Activity 5.
- **Model evaluation:** Record accuracy, precision, recall, F1-score, and the confusion matrix.
- **Real-time test:** Record the message you entered, predicted class, and both probabilities.

### Conclusion
This project applies probability and random-variable concepts to SMS spam detection. It calculates prior and conditional probabilities, verifies Bayes' theorem, and computes expectation and variance for a binary random variable. A Multinomial Naive Bayes classifier is then trained on word-count features and evaluated using standard classification metrics. Finally, the trained model is used to predict whether a newly entered SMS is spam or genuine.

## 11. GitHub submission checklist
The assignment asks you to update the project in a GitHub repository and share the repository link.

1. Download this notebook as an `.ipynb` file from the file link provided with this response.
2. Open [Google Colab](https://colab.research.google.com/) and upload the notebook.
3. Run every cell from top to bottom and check that the charts, tables, and metrics appear.
4. In GitHub, create a repository with a clear name such as `probabilistic-sms-spam-detection`.
5. Upload the completed `.ipynb` notebook. If required, add a `README.md` describing the project and how to run it.
6. Make sure no passwords, tokens, or other private information are included.
7. Copy the repository URL and submit/share it as required by your faculty.

**Important:** The notebook downloads the dataset when it is run, so you may need an internet connection in Colab.